In [5]:
%pip install requests pandas

Note: you may need to restart the kernel to use updated packages.


In [7]:
import requests
import pandas as pd

# The city's public map layer
LAYER = ("https://services5.arcgis.com/5RxyIIJ9boPdptdo/arcgis/rest/services/"
         "Housing_Tracker_Public_View_092122/FeatureServer/0/query")

# Same filter the dashboard uses. Change to "1=1" for every record.
WHERE = "(Stage = 'Stage 4') OR (Stage_5_Date > timestamp '2022-01-01 04:59:59')"

In [9]:
rows = []
offset = 0

while True:
    params = {
        "where": WHERE,
        "outFields": "*",         # every column
        "returnGeometry": "true",
        "outSR": 4326,            # latitude / longitude
        "resultOffset": offset,
        "resultRecordCount": 1000,
        "f": "json",
    }
    data = requests.get(LAYER, params=params, timeout=60).json()
    features = data.get("features", [])

    for f in features:
        record = dict(f["attributes"])
        geom = f.get("geometry") or {}
        record["longitude"] = geom.get("x")
        record["latitude"] = geom.get("y")
        rows.append(record)

    if not data.get("exceededTransferLimit") or not features:
        break
    offset += 1000

print("Downloaded", len(rows), "projects")

Downloaded 1419 projects


In [11]:
df = pd.DataFrame(rows)

# ArcGIS stores dates as milliseconds since 1970. Convert to real dates.
for col in df.columns:
    if "Date" in col or col.startswith("Month_and_Year"):
        df[col] = pd.to_datetime(df[col], unit="ms", errors="coerce")

# Put the most useful columns first
first = [c for c in ["Project_Name", "Project_Address_Display", "latitude", "longitude",
                     "Stage", "Project_Type", "Target_Population",
                     "Total_Units", "Number_of_Affordable_Units"] if c in df.columns]
df = df[first + [c for c in df.columns if c not in first]]

df.head()

,Project_Name,Project_Address_Display,latitude,longitude,Stage,Project_Type,Target_Population,Total_Units,Number_of_Affordable_Units,OBJECTID,...,Units_0_30__AMI,Units_31_50__AMI,Units_51_60__AMI,Units_61_80__AMI,Units_81_120__AMI,Units_Market_Rate,IZ_or_PSO_Project,IZ_Overlay,Ready_to_Display,GlobalID_2
0,Oglethorpe Place Apartments,"835 Oglethorpe Ave SW, Atlanta, GA 30310",33.736342,-84.415357,Stage 5,Rehab_HomeFlex,Renter,144,87,1,...,0.0,43.0,0.0,44.0,0.0,57.0,NaN,4%,,e207f7f9-17af-4865-b57c-434f79f028ce
1,1265 Lakewood,"1265 Lakewood Avenue SE, Atlanta, GA 30315",33.719533,-84.382966,Stage 5,New_Construction,Renter,160,160,6,...,0.0,26.0,121.0,13.0,0.0,0.0,Both,4%,,e6ac83d8-29fd-4300-a6f4-a0768043abc9
2,Ashley Scholars Landing IC,"152 Vine Street, Atlanta, GA 30314",33.750365,-84.407555,Stage 5,New_Construction,Renter,72,50,13,...,0.0,0.0,25.0,0.0,25.0,22.0,,NaN,,1eea9279-40af-4ef5-80b1-aae2383a5187
3,Lakewood Christian Manor,"2141 Springdale Road SW, Atlanta, GA 30315",33.696221,-84.414813,Stage 5,Rehab/Preservation,Renter,250,250,32,...,21.0,154.0,75.0,0.0,0.0,0.0,,4%,Yes,0fd5ceca-2961-4bf6-8437-2b0ced989973
4,Intrada Westside,"2174 Donald Lee Hollowell Parkway NW, Atlanta,...",33.775556,-84.457016,Stage 5,New_Construction,Renter,143,143,33,...,0.0,33.0,90.0,20.0,0.0,0.0,,4%,,d595e1bf-dcc9-4bf0-a9dd-4e6dcdf5fd5a


In [13]:
print(df.shape)                    # (rows, columns)
print(df["Stage"].value_counts())  # projects by stage
df.isna().sum().sort_values(ascending=False).head(10)  # most-missing columns

(1419, 55)
Stage
Stage 5    1356
Stage 4      63
Name: count, dtype: int64


IZ_Overlay                         1361
Project_Address_Display            1276
Supporting_Agencies__Other_        1270
Lead__Other_                       1270
TBD                                1251
IZ_or_PSO_Project                  1246
Project_Name                       1177
Other_Involved_Organizations       1057
City_Controlled_Public_Funding1     989
Closing_Date                        617
dtype: int64

In [15]:
df.to_csv("atlanta_affordable_housing_tracker.csv", index=False)
print("Saved!")

Saved!


In [17]:
stage5 = df[df["Stage"] == "Stage 5"]
stage4 = df[df["Stage"] == "Stage 4"]

print("Delivered affordable units:", stage5["Number_of_Affordable_Units"].sum())
print("Under construction affordable units:", stage4["Number_of_Affordable_Units"].sum())
print("Single-family units delivered:", (stage5["Total_Units"] == 1).sum())
print("Multifamily projects delivered:", (stage5["Total_Units"] > 1).sum())

Delivered affordable units: 8581
Under construction affordable units: 5493
Single-family units delivered: 1269
Multifamily projects delivered: 87


In [19]:
import pandas as pd

snap = pd.read_csv(r"C:\Users\Owner\Documents\GitHub\datasci530fall2026\Lecture 06\Historical SNAP Retailer Locator Data 2005-2025.csv", low_memory=False)
print(len(snap), "stores")
print(snap.columns.tolist())

703441 stores
['Record ID', 'Store Name', 'Store Type', 'Street Number', 'Street Name', 'Additional Address', 'City', 'State', 'Zip Code', 'Zip4', 'County', 'Latitude', 'Longitude', 'Authorization Date', 'End Date']


In [22]:
ga = snap[snap["State"] == "GA"].copy()
ga = ga[ga["End Date"].astype(str).str.strip() == ""]  # active stores have a blank End Date
print(len(ga), "active Georgia stores")
ga["Store Type"].value_counts()

9611 active Georgia stores


Store Type
Convenience Store            4943
Combination Grocery/Other    2459
Supermarket                   698
Super Store                   634
Medium Grocery Store          348
Small Grocery Store           153
Large Grocery Store            96
Meat/Poultry Specialty         84
Farmers' Market                77
Seafood Specialty              57
Fruits/Veg Specialty           37
Military Commissary             9
Delivery Route                  9
Bakery Specialty                6
Food Buying Co-op               1
Name: count, dtype: int64